# Loopo Italian FunctionGemma experiment

Use **Runtime → Change runtime type → T4 GPU** for the two-step smoke and full runs. On Hugging Face, accept the [FunctionGemma license](https://huggingface.co/google/functiongemma-270m-it), then grant this notebook access to a read token named **HF_TOKEN** in Colab Secrets. Run every cell in order. The default experiment scores a small validation slice and trains for two optimizer steps.

## 1. Environment and GPU diagnostics

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Select a Colab runtime with Python 3.11–3.13.")

print(sys.version)
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    check=False,
).returncode == 0
print("GPU available:", has_gpu)

## 2. Choose experiment, checkout and persistence

Use the ref containing this notebook. Keep `main` after merge or enter the feature branch or a commit SHA before merge. Use a unique run name for each fresh run. Set `resume = True` only for a saved checkpoint with identical settings. Mount Drive to keep checkpoints when Colab disconnects.

In [ ]:
#@title Experiment options
repo_ref = "main" #@param {type:"string"}
experiment = "smoke" #@param ["smoke", "seed", "balanced", "expanded", "base"]
use_drive = False #@param {type:"boolean"}
run_name = "trial-01" #@param {type:"string"}
resume = False #@param {type:"boolean"}
export_merged = False #@param {type:"boolean"}
examples_file = "" #@param {type:"string"}

if not has_gpu and experiment in {"seed", "balanced", "expanded", "base"}:
    raise RuntimeError("Select a GPU runtime for full training or held-out replay.")
if not run_name or Path(run_name).name != run_name or run_name in {".", ".."}:
    raise ValueError("Use a single directory name for run_name.")

checkout = Path("/content/loopo")
if not checkout.exists():
    subprocess.run(["git", "clone", "https://github.com/p33nah/loopo.git", str(checkout)], check=True)
subprocess.run(["git", "-C", str(checkout), "fetch", "origin", repo_ref], check=True)
subprocess.run(["git", "-C", str(checkout), "switch", "--detach", "FETCH_HEAD"], check=True)
project = checkout / "ml/functiongemma"
if not (project / "pyproject.toml").is_file():
    raise RuntimeError("Choose a repo_ref containing ml/functiongemma.")

results_root = Path("/content/loopo-runs")
if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    results_root = Path("/content/drive/MyDrive/loopo-functiongemma")
run = results_root / experiment / run_name
run.mkdir(parents=True, exist_ok=True)
print("Artifacts:", run)

## 3. Install pinned training dependencies

The notebook uses a virtual environment so the training packages do not replace Colab's kernel packages.

In [ ]:
venv = Path("/content/loopo-functiongemma-venv")
subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv)], check=True)
python = str(venv / "bin/python")
subprocess.run([sys.executable, "-m", "pip", "--python", python, "install", "pip"], check=True)
wheel_index = "cu128" if has_gpu else "cpu"
subprocess.run(
    [python, "-m", "pip", "install", "torch==2.10.0",
     "--index-url", f"https://download.pytorch.org/whl/{wheel_index}"],
    check=True,
)
subprocess.run([python, "-m", "pip", "install", "-e", str(project) + "[ml]"], check=True)
subprocess.run([python, "-m", "pip", "check"], check=True)
subprocess.run(
    [python, "-c", "import torch; print(torch.__version__, torch.version.cuda); "
     f"assert torch.cuda.is_available() == {has_gpu!r}"],
    check=True,
)

## 4. Authenticate with Hugging Face

The read token is passed to the ML subprocesses through their environment.

In [ ]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in Colab Secrets and enable notebook access.")
ml_env = {**os.environ, "HF_TOKEN": hf_token}
del hf_token

def cli(module, *args):
    subprocess.run(
        [python, "-m", "loopo_functiongemma." + module, *map(str, args)],
        cwd=project, env=ml_env, check=True,
    )

## 5. Generate and validate the dataset and tool schemas

In [ ]:
data = run / "data"
if experiment == "balanced":
    extra = ["--examples", examples_file] if examples_file else []
    cli("corpus", "generate", data, *extra)
else:
    expansion = ["--expand-train", "2500"] if experiment == "expanded" else []
    cli("dataset", "generate", data, *expansion)
cli("dataset", "validate", data)
subprocess.run(
    [python, "-c", "import json; from loopo_functiongemma.tools import TOOLS; print(json.dumps(TOOLS, indent=2, ensure_ascii=False))"],
    cwd=project, check=True,
)

## 6. Load the official tokenizer and verify native formatting

This step checks that every target begins after the inference prompt, fits the sequence length and round-trips to the expected ordered plan.

In [ ]:
cli("model", "validate-format", data)

## 7. Evaluate the base model

The smoke run uses four validation rows. Full and base-only experiments score the frozen test and challenge sets. Raw predictions are saved for the shared benchmark.

In [ ]:
if experiment == "smoke":
    sample = run / "smoke-validation.jsonl"
    lines = (data / "validation.jsonl").read_text(encoding="utf-8").splitlines()
    sample.write_text("\n".join(lines[:4]) + "\n", encoding="utf-8")
    benchmark_sets = {"smoke-validation": sample}
else:
    benchmark_sets = {split: data / f"{split}.jsonl" for split in ("test", "challenge")}

for name, input_file in benchmark_sets.items():
    cli("evaluate", "predict", input_file, run / f"{name}-base.json")

## 8. Fine-tune LoRA, select a checkpoint and save the adapter

The trainer uses validation loss to select the checkpoint. Compare training and validation loss in `training-metrics.json` or TensorBoard. For a new run choose a fresh run name. For a resume retain the same ref, dataset, config, GPU type and environment.

In [ ]:
training = run / "training"
config = project / "configs" / ("smoke.toml" if experiment == "smoke" else "train.toml")
if experiment != "base":
    cli("train", data, config, training, *(["--resume"] if resume else []))
    print((training / "training-metrics.json").read_text(encoding="utf-8"))

## 9. Evaluate and compare

The report contains machine-readable scores, per-example CSV, confusion counts and representative errors. The repository README shows how to add the Kotlin parser and classifier results for the same inputs.

In [ ]:
for name, input_file in benchmark_sets.items():
    report_inputs = ["--input", f"base={run / (name + '-base.json')}"]
    if experiment != "base":
        adapter_results = run / f"{name}-tuned.json"
        cli("evaluate", "predict", input_file, adapter_results, "--adapter", training / "adapter")
        report_inputs += ["--input", f"tuned={adapter_results}"]
    cli("evaluate", "compare", input_file, run / f"{name}-report", *report_inputs)
    print((run / f"{name}-report" / "summary.md").read_text(encoding="utf-8"))

## 10. Prepare a merged checkpoint for LiteRT-LM conversion

Enable `export_merged` to materialize standalone Hugging Face weights. The Android notes cover the separate conversion environment and acceptance checks.

In [ ]:
if export_merged and experiment != "base":
    cli("model", "merge", training / "adapter", run / "merged-hf")
print("Run path:", run)
for path in sorted(run.iterdir()):
    print(path)

## 11. Retain and reproduce the run

With Drive enabled the run is already on Drive. Otherwise download the run from the Files panel before ending the session. Retain the code ref, generated data, `training/run.json`, `training/training-metrics.json` and reports. Use the same ref and settings to reproduce the run. For the first full experiment choose `seed` with a fresh run name. Choose `balanced` and a fresh run name for the rejection-focused follow-up.